# NLP Multi-Label Emotion BERT Classifier Test File

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

In [2]:
# Load the training data.
train_df = pd.read_csv(
  "data/preprocessed/train.tsv", 
  sep="\t", 
  header=None, 
  names=["text", "emotion_ids", "id"]
)

train_df.head()

,text,emotion_ids,id
0,My favourite food is anything I didn't have to...,27,eebbqej
1,"Now if he does off himself, everyone will thin...",27,ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj
3,To make her feel threatened,14,ed7ypvh
4,Dirty Southern Wankers,3,ed0bdzj


In [3]:
print(train_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [4]:
print(train_df.shape)

(43410, 3)


In [6]:
val_df = pd.read_csv(
  "data/preprocessed/dev.tsv", 
  sep="\t", 
  header=None, 
  names=["text", "emotion_ids", "id"]
)

val_df.head()

,text,emotion_ids,id
0,Is this in New Orleans?? I really feel like th...,27,edgurhb
1,"You know the answer man, you are programmed to...","4,27",ee84bjg
2,I've never been this sad in my life!,25,edcu99z
3,The economy is heavily controlled and subsidiz...,"4,27",edc32e2
4,He could have easily taken a real camera from ...,20,eepig6r


In [7]:
print(val_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [8]:
print(val_df.shape)

(5426, 3)


In [18]:
# Load in test.
test_df = pd.read_csv(
    "data/preprocessed/test.tsv",
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"]
)

In [19]:
print(test_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [20]:
print(test_df.shape)

(5427, 3)


In [9]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43410 entries, 0 to 43409
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   text         43410 non-null  str  
 1   emotion_ids  43410 non-null  str  
 2   id           43410 non-null  str  
dtypes: str(3)
memory usage: 4.2 MB


In [10]:
train_df['emotion_ids'].value_counts()

emotion_ids
27            12823
0              2710
4              1873
15             1857
1              1652
              ...  
0,12,13,26        1
1,2,5,17          1
13,14             1
3,9,12            1
0,1,18            1
Name: count, Length: 711, dtype: int64

In [11]:
train_df['emotion_ids'].value_counts().head(20)

emotion_ids
27    12823
0      2710
4      1873
15     1857
1      1652
3      1451
18     1427
10     1402
7      1389
2      1025
20      861
6       858
17      853
25      817
26      720
9       709
5       649
22      586
13      510
11      498
Name: count, dtype: int64

In [12]:
train_df['emotion_ids'].nunique()

711

In [14]:
train_df['emotion_ids'].unique()[:30]

<ArrowStringArray>
[    '27',      '2',     '14',      '3',     '26',     '15',   '8,20',
      '0',      '6',    '1,4',      '5',   '3,12',   '6,22', '6,9,27',
     '12',  '16,25',    '2,7',     '17',     '25',   '0,15',  '15,18',
  '16,27',   '7,13',     '10',     '20',      '4',  '13,15',    '0,1',
     '13',      '1']
Length: 30, dtype: str

In [15]:
train_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    36308
True      7102
Name: count, dtype: int64

In [16]:
total_count = len(train_df)
multi_labeled_count = train_df['emotion_ids'].str.contains(',').sum()
single_labeled_count = (~train_df['emotion_ids'].str.contains(',')).sum()

print(f"Total data points: {total_count}")
print(f"Single-labeled data points: {single_labeled_count}")
print(f"Multi-labeled data points: {multi_labeled_count}")

Total data points: 43410
Single-labeled data points: 36308
Multi-labeled data points: 7102


In [17]:
val_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    4548
True      878
Name: count, dtype: int64

In [21]:
test_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    4590
True      837
Name: count, dtype: int64

## Working with Multi-Labeled Data

In [22]:
from sklearn.preprocessing import MultiLabelBinarizer

In [23]:
emotion_names = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

In [24]:
train_df['emotion_id_list'] = (
  train_df['emotion_ids'].str.split(',')
  .apply(lambda ids: [int(i) for i in ids])
)

In [26]:
train_df.tail()

,text,emotion_ids,id,emotion_id_list
43405,Added you mate well I’ve just got the bow and ...,18,edsb738,[18]
43406,Always thought that was funny but is it a refe...,6,ee7fdou,[6]
43407,What are you talking about? Anything bad that ...,3,efgbhks,[3]
43408,"More like a baptism, with sexy results!",13,ed1naf8,[13]
43409,Enjoy the ride!,17,eecwmbq,[17]


In [27]:
mlb = MultiLabelBinarizer(classes=range(len(emotion_names)))

y = mlb.fit_transform(train_df['emotion_id_list'])

print(y.shape)

(43410, 28)


In [28]:
train_df[train_df['emotion_ids'].str.contains(',')]

,text,emotion_ids,id,emotion_id_list
7,We need more boards and to create a bit more s...,"8,20",ef4qmod,"[8, 20]"
11,"Aww... she'll probably come around eventually,...","1,4",edex4ki,"[1, 4]"
15,"Shit, I guess I accidentally bought a Pay-Per-...","3,12",edivtm3,"[3, 12]"
19,Maybe that’s what happened to the great white ...,"6,22",eczq8zg,"[6, 22]"
20,"I never thought it was at the same moment, but...","6,9,27",efdlhs1,"[6, 9, 27]"
...,...,...,...,...
43382,"goat handshake denied Personally, I just thin...","14,27",ef2m53x,"[14, 27]"
43383,it's horrid :/,"14,27",edlbr3j,"[14, 27]"
43388,Fuck these trendy hipster joints. Give me my s...,"2,3",ee3nyiy,"[2, 3]"
43395,Sorry I kind of took it like you were flexing ...,"1,24",eelhhzc,"[1, 24]"


In [29]:
y[:1]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1]])

In [30]:
y[43382:43384]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1]])

In [31]:
# Same parsing you used for train_df
val_df["emotion_id_list"] = (
    val_df["emotion_ids"]
    .str.split(",")
    .apply(lambda ids: [int(i) for i in ids])
)

In [32]:
val_df.tail()

,text,emotion_ids,id,emotion_id_list
5421,It's pretty dangerous when the state decides w...,14,edyrazk,[14]
5422,I filed for divorce this morning. Hoping he mo...,20,edi2z3y,[20]
5423,"The last time it happened I just said, ""No"" an...",10,eewbqtx,[10]
5424,I can’t stand this arrogant prick he’s no bett...,3,eefx57m,[3]
5425,::but I like baby bangs:: /tiny voice,18,ed5h3jh,[18]


In [33]:
y_val = mlb.transform(val_df["emotion_id_list"])
print(y_val)

[[0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [34]:
test_df["emotion_id_list"] = (
    test_df["emotion_ids"]
    .str.split(",")
    .apply(lambda ids: [int(i) for i in ids])
)

In [35]:
test_df.tail()

,text,emotion_ids,id,emotion_id_list
5422,Thanks. I was diagnosed with BP 1 after the ho...,15,efeeasc,[15]
5423,Well that makes sense.,4,ef9c7s3,[4]
5424,Daddy issues [NAME],27,efbiugo,[27]
5425,So glad I discovered that subreddit a couple m...,0,efbvgp9,[0]
5426,"Had to watch ""Elmo in Grouchland"" one time too...",27,edtjpv6,[27]


In [36]:
print(test_df.shape)

(5427, 4)


In [37]:
y_test = mlb.transform(test_df["emotion_id_list"])

In [38]:
print(y_test.shape)

(5427, 28)


## BERT Model

In [39]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch 
from torch.utils.data import Dataset, DataLoader

In [41]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [42]:
text = "I love this game"

tokens = tokenizer.tokenize(text)

print(tokens)

['i', 'love', 'this', 'game']


In [43]:
token_ids = tokenizer.encode(text)
print(token_ids)

[101, 1045, 2293, 2023, 2208, 102]


In [44]:
train_df.head()

,text,emotion_ids,id,emotion_id_list
0,My favourite food is anything I didn't have to...,27,eebbqej,[27]
1,"Now if he does off himself, everyone will thin...",27,ed00q6i,[27]
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj,[2]
3,To make her feel threatened,14,ed7ypvh,[14]
4,Dirty Southern Wankers,3,ed0bdzj,[3]


In [45]:
X_train = train_df["text"].tolist()
X_train[:10]

["My favourite food is anything I didn't have to cook myself.",
 'Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actually dead',
 'WHY THE FUCK IS BAYLESS ISOING',
 'To make her feel threatened',
 'Dirty Southern Wankers',
 "OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe PlAyOfFs! Dumbass Broncos fans circa December 2015.",
 'Yes I heard abt the f bombs! That has to be why. Thanks for your reply:) until then hubby and I will anxiously wait 😝',
 'We need more boards and to create a bit more space for [NAME]. Then we’ll be good.',
 'Damn youtube and outrage drama is super lucrative for reddit',
 'It might be linked to the trust factor of your friend.']

In [47]:
X_val = val_df["text"].tolist()
X_val[:5]

['Is this in New Orleans?? I really feel like this is New Orleans.',
 'You know the answer man, you are programmed to capture those codes they send you, don’t avoid them!',
 "I've never been this sad in my life!",
 'The economy is heavily controlled and subsidized by the government. In any case, I was poking at the lack of nuance in US politics today',
 'He could have easily taken a real camera from a legitimate source and change the price in Word/Photoshop and then print it out.']

In [48]:
X_test = test_df["text"].tolist()
X_test[:5]

['I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!',
 "It's wonderful because it's awful. At not with.",
 'Kings fan here, good luck to you guys! Will be an interesting game to watch! ',
 "I didn't know that, thank you for teaching me something today!",
 'They got bored from haunting earth for thousands of years and ultimately moved on to the afterlife.']

In [55]:
MAX_LEN = 64

class GoEmotionsDataset(Dataset):
  
  def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
    self.texts = texts 
    self.labels = labels
    self.tokenizer = tokenizer 
    self.max_len = max_len
    
  def __len__(self):
    return len(self.texts)
  
  def __getitem__(self, idx):
    encoding = self.tokenizer(
      self.texts[idx], 
      truncation=True, 
      padding="max_length", 
      max_length=self.max_len, 
      return_tensors="pt"
    )
    
    return {
      "input_ids": encoding["input_ids"].squeeze(0), 
      "attention_mask": encoding["attention_mask"].squeeze(0), 
      "labels": torch.tensor(self.labels[idx], dtype=torch.float) # float, not long — needed for BCE loss
    }

In [56]:
train_dataset = GoEmotionsDataset(X_train, y, tokenizer)
val_dataset = GoEmotionsDataset(X_val, y_val, tokenizer)
test_dataset = GoEmotionsDataset(X_test, y_test, tokenizer)

In [57]:
print(type(train_dataset))
print(len(train_dataset))

print(train_dataset.texts[0])

<class '__main__.GoEmotionsDataset'>
43410
My favourite food is anything I didn't have to cook myself.


In [60]:
train_dataset[0]

{'input_ids': tensor([ 101, 2026, 8837, 2833, 2003, 2505, 1045, 2134, 1005, 1056, 2031, 2000,
         5660, 2870, 1012,  102,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'labels': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])}

In [61]:
print(type(val_dataset))
print(len(val_dataset))
print(val_dataset.texts[0])

<class '__main__.GoEmotionsDataset'>
5426
Is this in New Orleans?? I really feel like this is New Orleans.


In [62]:
print(type(test_dataset))
print(len(test_dataset))
print(test_dataset.texts[0])

<class '__main__.GoEmotionsDataset'>
5427
I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!


In [63]:
# DataLoader turns Dataset (only knows how to return one example at a time via __getitem__) into something that hands back BATCHES of examples.

# DataLoder is grouping 16 examples into a batch.

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [65]:
batch = next(iter(train_loader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 64])
torch.Size([16, 64])
torch.Size([16, 28])


In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(emotion_names),  # 28
    problem_type="multi_label_classification"
)
model.to(device)

cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6276.02it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [67]:
from torch.optim import AdamW 
from torch.utils.data import Subset 
import time

In [68]:
# --- Optional: quick subset run first to validate the loop works ---
DEBUG_SUBSET = True  # set False once you've confirmed everything runs cleanly

if DEBUG_SUBSET:
    small_train = Subset(train_dataset, range(2000))
    small_val = Subset(val_dataset, range(500))
    train_loader = DataLoader(small_train, batch_size=8, shuffle=True)
    val_loader = DataLoader(small_val, batch_size=8)
else:
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    start = time.time()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{EPOCHS} — avg loss: {avg_loss:.4f} — time: {elapsed/60:.1f} min")

    # Save after each epoch in case something crashes later
    model.save_pretrained(f"bert_multilabel_epoch{epoch+1}")
    tokenizer.save_pretrained(f"bert_multilabel_epoch{epoch+1}")

Epoch 1/2 — avg loss: 0.2526 — time: 2.4 min


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]


Epoch 2/2 — avg loss: 0.1492 — time: 2.2 min


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]


In [69]:
full_train_size = 43410
subset_size = 2000
ratio = full_train_size / subset_size  # ~21.7

epoch1_est = 2.3 * ratio
epoch2_est = 2.1 * ratio
print(f"Estimated epoch 1: {epoch1_est:.0f} min ({epoch1_est/60:.1f} hr)")
print(f"Estimated epoch 2: {epoch2_est:.0f} min ({epoch2_est/60:.1f} hr)")
print(f"Estimated total (2 epochs): {(epoch1_est+epoch2_est)/60:.1f} hr")

Estimated epoch 1: 50 min (0.8 hr)
Estimated epoch 2: 46 min (0.8 hr)
Estimated total (2 epochs): 1.6 hr
